<a href="https://colab.research.google.com/github/Naufallm/RAG-Quantitative-Research-Engine-for-IDX/blob/main/DSC_Project_IDX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Member
- Ahmad Naufal Luthfan Marzuqi
- Syahrial Nur Faturrahman

# Step 1: Install Library
Pada tahap ini, kita menginstal library `pypdf` untuk membaca file PDF dan `langchain` untuk memproses teks.
Library `pandas` digunakan untuk manajemen data tabel jika diperlukan.

Pada Step ini akan diminta untuk restart runtime, setelah restart dapat dijalankan kembali mulai dari awal ya!


In [ ]:
!pip install -U pypdf langchain langchain-community langchain-text-splitters pandas

# Step 2: Sinkronisasi Data
Kita mengambil dataset laporan keuangan yang sudah dikumpulkan di GitHub agar bisa diproses langsung di environment Colab ini.


Untuk link github nya https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX

In [ ]:
!git clone https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX.git

fatal: destination path 'RAG-Quantitative-Research-Engine-for-IDX' already exists and is not an empty directory.


# Step 3: Verifikasi File PDF
Tahap ini memastikan bahwa folder `dataset_idx` telah terunduh dengan benar dan menghitung jumlah file yang siap diproses.

In [ ]:
import os

dataset_path = "/content/RAG-Quantitative-Research-Engine-for-IDX/dataset_idx"
pdf_files = [f for f in os.listdir(dataset_path) if f.endswith(".pdf")]
print("Jumlah file PDF:", len(pdf_files))
print("\nDaftar file PDF: ")
for file in pdf_files:
  print(file)

Jumlah file PDF: 50

Daftar file PDF: 
FinancialStatement-2026-I-IBFN.pdf
FinancialStatement-2026-I-BOBA.pdf
FinancialStatement-2026-I-PMJS.pdf
FinancialStatement-2026-I-IKPM.pdf
FinancialStatement-2026-I-EDGE.pdf
FinancialStatement-2026-I-BLOG.pdf
FinancialStatement-2026-I-GUNA.pdf
FinancialStatement-2026-I-BINA.pdf
FinancialStatement-2026-I-BANK.pdf
FinancialStatement-2026-I-BOLT.pdf
FinancialStatement-2026-I-IPCC.pdf
FinancialStatement-2026-I-BMRI.pdf
FinancialStatement-2026-I-SDRA.pdf
FinancialStatement-2026-I-MYTX.pdf
FinancialStatement-2026-I-AGRO.pdf
FinancialStatement-2026-I-BEKS.pdf
FinancialStatement-2026-I-DSNG.pdf
FinancialStatement-2026-I-MCOL.pdf
FinancialStatement-2026-I-PGLI.pdf
FinancialStatement-2026-I-STAA.pdf
FinancialStatement-2026-I-BBCA.pdf
FinancialStatement-2026-I-SNLK.pdf
FinancialStatement-2026-I-RISE.pdf
FinancialStatement-2026-I-BBSS.pdf
FinancialStatement-2026-I-MSIE.pdf
FinancialStatement-2026-I-FORE.pdf
FinancialStatement-2026-I-PMUI.pdf
FinancialStateme

# Step 4: PDF Text Extraction
Di sini kita membaca setiap halaman PDF dan mengubahnya menjadi teks string.
Kita juga mengambil Nama Perusahaan (Ticker) dari nama file menggunakan fungsi split teks.

In [ ]:
from pypdf import PdfReader

all_raw_documents = []

for filename in pdf_files:
    path = os.path.join(dataset_path, filename)
    try:
        reader = PdfReader(path)
        full_text = ""
        for page in reader.pages:
            page_content = page.extract_text()
            if page_content:
                full_text += page_content + "\n"

        ticker_name = filename.split("-")[-1].replace(".pdf", "")

        all_raw_documents.append({
            "ticker": ticker_name,
            "filename": filename,
            "text": full_text
        })
    except Exception as e:
        print(f"Gagal membaca {filename}: {e}")

print(f"Proses selesai. Berhasil mengekstrak {len(all_raw_documents)} dokumen.")

Proses selesai. Berhasil mengekstrak 50 dokumen.


# Step 5: Hierarchical Chunking (Fixed Version)
Kita menggunakan paket 'langchain_text_splitters' yang merupakan standar terbaru dari LangChain.
Teks laporan keuangan akan dipecah menjadi bagian-bagian kecil agar dapat diproses secara efisien
oleh model bahasa (Groq/LLM).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Inisialisasi splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

final_chunks = []

# Proses pemotongan teks dari dokumen yang sudah diekstrak di Step 4
for doc in all_raw_documents:
    chunks = text_splitter.split_text(doc["text"])
    for chunk in chunks:
        final_chunks.append({
            "ticker": doc["ticker"],
            "source": doc["filename"],
            "content": chunk
        })

print(f"Total chunks yang dihasilkan: {len(final_chunks)}")

Total chunks yang dihasilkan: 9744


# Step 6: Quality Control & Sampel Output
Tahap terakhir adalah memastikan data hasil potongan (chunk) memiliki metadata yang benar
dan menampilkan sampel isi datanya untuk bahan laporan checkpoint.

In [ ]:
print(f"Total File Berhasil Diproses : {len(all_raw_documents)}")
print(f"Total Potongan Teks (Chunks) : {len(final_chunks)}")
print("-" * 40)

if len(final_chunks) > 0:
    print(f"SAMPEL DATA DARI EMITEN     : {final_chunks[0]['ticker']}")
    print(f"SUMBER FILE ASLI            : {final_chunks[0]['source']}")
    print("-" * 40)
    print("CUPLIKAN ISI TEKS:")
    print(final_chunks[0]['content'][:600] + "...")

Total File Berhasil Diproses : 50
Total Potongan Teks (Chunks) : 9744
----------------------------------------
SAMPEL DATA DARI EMITEN     : IBFN
SUMBER FILE ASLI            : FinancialStatement-2026-I-IBFN.pdf
----------------------------------------
CUPLIKAN ISI TEKS:
Perseroan dengan ini menyampaikan laporan keuangan untuk periode 3 Bulan yang berakhir pada 31/03/2026 dengan ikhtisar sebagai berikut : 
  
Informasi mengenai anak perusahaan Perseroan sebagai berikut : 
  
  
  
  
Dokumen ini merupakan dokumen resmi PT Intan Baru Prana Tbk yang tidak memerlukan tanda tangan karena dihasilkan secara elektronik. PT
Intan Baru Prana Tbk bertanggung jawab penuh atas informasi tertera di dalam dokumen ini. 
Nomor Surat 004/ACCT-IBP/IV/2026
Nama Emiten PT Intan Baru Prana Tbk
Kode Emiten IBFN
Perihal Penyampaian Laporan Keuangan Interim Yang Tidak Diaudit
[100000...


# Step 7: Instalasi Library Database & Performance Monitoring
Menginstal library yang dibutuhkan untuk:
- `chromadb` → penyimpanan database vektor
- `sentence-transformers` → model embedding teks
- `langchain-huggingface` → integrasi Hugging Face dengan LangChain
- `tqdm` → progress bar saat proses berjalan

Jangan lupa setelah install, di import ya

In [ ]:
# tulis code disini

# Step 8: Pemilihan Model Embedding Multilingual
Laporan keuangan IDX bersifat bilingual. Kita menggunakan model `paraphrase-multilingual-MiniLM-L12-v2` dalam memetakan kesamaan makna antara Bahasa Indonesia dan Inggris.

In [ ]:
# tulis code disini

# Step 9: Konstruksi Persistent ChromaDB
Kita akan memasukkan 9.744 chunks ke dalam database.
Berbeda dengan database biasa, kita mengaktifkan `persist_directory` agar data tersimpan di penyimpanan permanen.
Kita menggunakan batching untuk menjaga kestabilan memori Colab.

In [ ]:
# tulis code disini

# Step 10: Audit Integritas Database
Sebagai Quality Control, kita akan mengecek apakah semua ticker perusahaan sudah masuk ke database
dan berapa jumlah potongan teks (chunks) per perusahaan.

In [ ]:
# tulis code disini

# Step 11: Pengujian Pencarian Cerdas (MMR Search)
Kita akan menguji pencarian menggunakan teknik MMR (Maximum Marginal Relevance).
Teknik ini memastikan hasil pencarian relevan namun tetap bervariasi (tidak memberikan jawaban yang itu-itu saja).

In [ ]:
# tulis code disni